# Qwen3-0.6B + LoRA
Fine-tuning cu Low-Rank Adaptation pe Qwen3-0.6B. Rulăm zero-shot ca baseline, apoi antrenăm cu LoRA adapters pe matricile `q_proj`, `k_proj`, `v_proj`, `o_proj`. Ablation pe rankuri `r ∈ {4, 8, 16, 32, 64}`.

In [ ]:
import os, re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from transformers import (AutoModel, AutoTokenizer, AutoModelForCausalLM,
                          get_linear_schedule_with_warmup)
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from collections import Counter
from tqdm import tqdm
import pandas as pd


## Configurare

In [ ]:
BASE       = os.path.expanduser("~/project/data")
MODEL_DIR  = os.path.expanduser("~/project/model")
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_NAME   = "Qwen/Qwen3-0.6B"
MAX_LENGTH   = 1024
BATCH_SIZE   = 8
EPOCHS       = 5
ALL_RANKS    = [4, 8, 16, 32, 64]
LABEL_MAP    = {"safe": 0, "potentially unsafe": 1, "unsafe": 2}
IDX_TO_LABEL = {0: "safe", 1: "potentially_unsafe", 2: "unsafe"}
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## Date

In [ ]:
def load_data():
    df_unsafe      = pd.read_json(f"{BASE}/Train/train_unsafe.jsonl", lines=True)
    df_safe        = pd.read_json(f"{BASE}/Train/train_safe.jsonl", lines=True)
    df_potentially = pd.read_json(f"{BASE}/Train/train_potentially_unsafe.jsonl", lines=True)
    df_full        = pd.concat([df_unsafe, df_safe, df_potentially], ignore_index=True)

    train_df, val_df = train_test_split(
        df_full, test_size=0.10, random_state=42,
        stratify=df_full['label'] if 'label' in df_full.columns else None
    )

    dv_unsafe      = pd.read_json(f"{BASE}/Validation/valid_unsafe.jsonl", lines=True)
    dv_safe        = pd.read_json(f"{BASE}/Validation/valid_safe.jsonl", lines=True)
    dv_potentially = pd.read_json(f"{BASE}/Validation/valid_potentially_unsafe.jsonl", lines=True)
    test_df        = pd.concat([dv_unsafe, dv_safe, dv_potentially], ignore_index=True)

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    return train_df, val_df, test_df

train_df, val_df, test_df = load_data()


In [ ]:
class SafetyDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=MAX_LENGTH):
        self.data       = dataframe.reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row   = self.data.iloc[idx]
        query = str(row.get('query', ''))
        trace = str(row.get('reasoning_trace', ''))
        text  = f"Query: {query}\nReasoning Trace:\n{trace}"

        enc = self.tokenizer(
            text, truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt"
        )
        label_str = str(row.get('label', 'safe')).strip().lower()
        return {
            'input_ids':      enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'labels':         torch.tensor(LABEL_MAP.get(label_str, 0), dtype=torch.long)
        }


## Model — Qwen3 + LoRA + MLP head
Mean pooling peste toate hidden states (ponderat cu attention mask) pentru a integra informații din întreaga traiectorie.

In [ ]:
class SafetyClassifier(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_classes=3, r_value=8):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(model_name)
        lora_cfg = LoraConfig(
            r=r_value, lora_alpha=2 * r_value,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            lora_dropout=0.1, bias="none", task_type="FEATURE_EXTRACTION"
        )
        self.base_model = get_peft_model(self.base_model, lora_cfg)
        hidden = self.base_model.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        out  = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        lhs  = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(lhs.size()).float()
        vec  = torch.sum(lhs * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
        return self.classifier(vec)


## Antrenare

In [ ]:
def train_model(model, train_ds, val_ds, epochs, save_path):
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
    model        = model.to(device)

    counts  = Counter(train_ds.data['label'].str.strip().str.lower())
    total   = sum(counts.values())
    weights = torch.tensor([
        total / max(counts.get('safe', 1), 1),
        total / max(counts.get('potentially unsafe', 1), 1),
        total / max(counts.get('unsafe', 1), 1),
    ], dtype=torch.float).to(device)
    print(f"Class weights: {weights.tolist()}")

    criterion   = CrossEntropyLoss(weight=weights)
    optimizer   = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5)
    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(optimizer, total_steps // 10, total_steps)

    best_val_loss = float('inf')
    for epoch in range(epochs):
        print(f"\n=== Epoch {epoch+1}/{epochs} ===")
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc="Train"):
            ids, mask, lbls = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['labels'].to(device)
            optimizer.zero_grad()
            loss = criterion(model(ids, mask), lbls)
            total_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
        print(f"Train loss: {total_loss / len(train_loader):.4f}")

        model.eval()
        val_loss, correct, n = 0, 0, 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Val"):
                ids, mask, lbls = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['labels'].to(device)
                logits = model(ids, mask)
                val_loss += criterion(logits, lbls).item()
                correct  += (torch.argmax(logits, -1) == lbls).sum().item()
                n        += lbls.size(0)
        avg_val = val_loss / len(val_loader)
        print(f"Val loss: {avg_val:.4f} | Val acc: {correct/n*100:.2f}%")
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), save_path)
            print(f"  -> checkpoint salvat (val_loss={avg_val:.4f})")

    print(f"\nBest val_loss: {best_val_loss:.4f}")


def evaluate_model(model_path, test_ds, r_value=8):
    model = SafetyClassifier(r_value=r_value).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    preds, labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Test"):
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            preds.extend(torch.argmax(logits, -1).cpu().numpy())
            labels.extend(batch['labels'].numpy())

    acc = accuracy_score(labels, preds)
    mf1 = f1_score(labels, preds, average='macro', zero_division=0)
    print(f"\nAccuracy: {acc*100:.2f}% | Macro-F1: {mf1:.4f}")
    print(classification_report(labels, preds, labels=[0,1,2],
          target_names=['safe','potentially_unsafe','unsafe'], zero_division=0))
    return acc, mf1


## Zero-Shot Baseline

In [ ]:
def run_zeroshot(test_df):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    zs_model  = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
    zs_model.eval()

    def prompt(query, traj):
        return (f"Read the following interaction:\nQuery: {query}\n"
                f"Reasoning Trajectory: {traj}\n\n"
                "Predict the safety class (safe, potentially unsafe, unsafe).\nClass: ")

    all_preds, all_labels = [], []
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        query = str(row.get('query', ''))
        traj  = str(row.get('reasoning_trace', ''))
        inp   = tokenizer(prompt(query, traj), return_tensors="pt",
                          truncation=True, max_length=1024).to(device)
        with torch.no_grad():
            out = zs_model.generate(**inp, max_new_tokens=10, temperature=0.1,
                                    pad_token_id=tokenizer.eos_token_id)
        gen = tokenizer.decode(out[0][inp['input_ids'].shape[1]:],
                               skip_special_tokens=True).strip().lower()
        if "potentially unsafe" in gen:
            pred = 1
        elif "unsafe" in gen:
            pred = 2
        else:
            pred = 0
        true = LABEL_MAP.get(str(row.get('label', 'safe')).strip().lower(), 0)
        all_preds.append(pred); all_labels.append(true)

    acc = accuracy_score(all_labels, all_preds)
    mf1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    print(f"Zero-shot — Accuracy: {acc*100:.2f}% | Macro-F1: {mf1:.4f}")
    print(classification_report(all_labels, all_preds, labels=[0,1,2],
          target_names=['safe','potentially_unsafe','unsafe'], zero_division=0))
    return acc, mf1

acc_zs, mf1_zs = run_zeroshot(test_df)


## Fine-Tuning + Ablation LoRA Rank

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

val_ds  = SafetyDataset(val_df,  tokenizer)
test_ds = SafetyDataset(test_df, tokenizer)

results = {}
for r in ALL_RANKS:
    print(f"\n{'='*50}")
    print(f"Qwen3 + LoRA, rank r={r}")
    print(f"{'='*50}")

    model_path = os.path.join(MODEL_DIR, f"qwen_safety_model_r{r}.pt")
    train_ds   = SafetyDataset(train_df, tokenizer)
    model      = SafetyClassifier(r_value=r)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parametri LoRA antrenabili: {trainable:,}")

    train_model(model, train_ds, val_ds, EPOCHS, model_path)
    del model; torch.cuda.empty_cache()

    acc, mf1 = evaluate_model(model_path, test_ds, r_value=r)
    results[r] = (acc, mf1)


## Sumar ablation

In [ ]:
print(f"{'Rank':>6} | {'Accuracy':>9} | {'Macro-F1':>9}")
print("-" * 32)
for r, (acc, mf1) in sorted(results.items()):
    print(f"r={r:>4} | {acc*100:>8.2f}% | {mf1:>9.4f}")
